In [4]:
import numpy as np
from collections import defaultdict

class NaiveBayesClassifier:

    def fit(self, X_text, y):
        """
        X_text: list of strings (emails)
        y: list of 0/1 labels (0=ham, 1=spam)
        """
        self.classes   = [0, 1]
        self.log_prior = {}
        self.log_likelihood = {}

        n = len(y)
        
        for c in self.classes:
            docs = [X_text[i] for i in range(n) if y[i] == c]

            self.log_prior[c] = np.log(len(docs) / n)

        
            word_counts = defaultdict(int)
            total_words = 0
            for doc in docs:
                for word in doc.lower().spill:
                    word_counts[word] += 1
                    total_words       += 1
                    
            vocab_size = len(set(
                w for doc in X_text
                for w in doc.lower().split()
            ))
            self.log_likelihood[c] = {
                word: np.log((count + 1) /
                             (total_words + vocab_size))
                for word, count in word_counts.items()
            }
            # default for unseen words
            self.default_ll = {
                c: np.log(1 / (total_words + vocab_size))
                for c in self.classes
            }
        return self
    
    def predict_one(self, text):
        scores = {}
        for c in self.classes:
            # start with log prior
            score = self.log_prior[c]
            # add log likelihood of each word
            for word in text.lower().split():
                score += self.log_likelihood[c].get(
                    word, self.default_ll[c]
                )
            scores[c] = score

        return max(scores, key=scores.get)
    
    def predict(self, X_text):
        return [self.predict_one(doc) for doc in X_text]

    def score(self, X_text, y):
        preds = self.predict(X_text)
        return sum(p == t for p, t in zip(preds, y)) / len(y)
    

In [ ]:
# Training data — small email dataset
emails = [
    # spam
    "free money win prize now",
    "click here free offer limited time",
    "congratulations you won free cash",
    "buy now discount free shipping offer",
    "win lottery free ticket claim prize",
    "free gift card click now limited offer",
    "urgent free money transfer available",
    "exclusive deal free prize winner",

    # ham
    "meeting at 3pm tomorrow office",
    "can you review the project report",
    "lunch today at the cafeteria",
    "please send the quarterly report",
    "team standup tomorrow morning",
    "the code review is scheduled friday",
    "happy birthday have a great day",
    "reminder about the budget meeting",
]

labels = [1,1,1,1,1,1,1,1,  # spam
          0,0,0,0,0,0,0,0]  # ham


# Train
nb = NaiveBayesClassifier()
nb.fit(emails, labels)
print(f"Training accuracy: {nb.score(emails, labels):.2f}")

test_emails = [
    "free prize click now",           # should be spam
    "project meeting tomorrow 3pm",   # should be ham
    "free lunch meeting today",       # ambiguous — what do you think?
    "win free report quarterly",      # ambiguous
]

for email in test_emails:
    pred = nb.predict_one(email)
    label = "SPAM" if pred == 1 else "HAM"
    print(f"{label}: '{email}'")

AttributeError: 'str' object has no attribute 'spill'